### Build Drivers Dimension

In [0]:
%run ../00-common/01-environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
from pyspark.sql import functions as F

### 1. Read source tables
silver.drivers, gold.ref_natioanlity_region

In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers")
nationality_region_ref_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
display(drivers_df)
display(nationality_region_ref_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
alesi,1964-06-11,French,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jean Alesi
ashdown,1934-10-16,British,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Peter Ashdown
baghetti,1934-12-25,Italian,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Giancarlo Baghetti
bauer,1912-07-17,German,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Erwin Bauer
belmondo,1963-04-23,French,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Paul Belmondo
bianco,1916-07-22,Brazilian,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Gino Bianco
binder,1948-06-12,Austrian,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Hans Binder
bira,1914-07-15,Thai,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Prince Bira
bonetto,1903-06-09,Italian,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Felice Bonetto
brancatelli,1950-01-18,Italian,2026-07-28T10:52:43.798Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Gianfranco Brancatelli


nationality,region
British,Europe
Italian,Europe
French,Europe
German,Europe
Swiss,Europe
Dutch,Europe
Belgium,Europe
Belgian,Europe
Irish,Europe
Spanish,Europe


### 2. Join two tables

In [0]:
dim_driver_df = (
    drivers_df
    .join(nationality_region_ref_df,
            on = "nationality",
            how = "left")
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        nationality_region_ref_df.region.alias("nationality_region")
    )
)
display(dim_driver_df)

driver_id,driver_name,date_of_birth,nationality,nationality_region
alesi,Jean Alesi,1964-06-11,French,Europe
ashdown,Peter Ashdown,1934-10-16,British,Europe
baghetti,Giancarlo Baghetti,1934-12-25,Italian,Europe
bauer,Erwin Bauer,1912-07-17,German,Europe
belmondo,Paul Belmondo,1963-04-23,French,Europe
bianco,Gino Bianco,1916-07-22,Brazilian,South America
binder,Hans Binder,1948-06-12,Austrian,Europe
bira,Prince Bira,1914-07-15,Thai,Asia
bonetto,Felice Bonetto,1903-06-09,Italian,Europe
brancatelli,Gianfranco Brancatelli,1950-01-18,Italian,Europe


### Writing data to gold table

In [0]:
(
    dim_driver_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(target_table)
    )
display(spark.table(target_table))

driver_id,driver_name,date_of_birth,nationality,nationality_region
alesi,Jean Alesi,1964-06-11,French,Europe
ashdown,Peter Ashdown,1934-10-16,British,Europe
baghetti,Giancarlo Baghetti,1934-12-25,Italian,Europe
bauer,Erwin Bauer,1912-07-17,German,Europe
belmondo,Paul Belmondo,1963-04-23,French,Europe
bianco,Gino Bianco,1916-07-22,Brazilian,South America
binder,Hans Binder,1948-06-12,Austrian,Europe
bira,Prince Bira,1914-07-15,Thai,Asia
bonetto,Felice Bonetto,1903-06-09,Italian,Europe
brancatelli,Gianfranco Brancatelli,1950-01-18,Italian,Europe
